In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.svm import OneClassSVM

project_path = '/content/drive/MyDrive/Spacecraft-Anomaly-Detection'

data_path = project_path + '/data/raw/archive/data/data'
train_path = data_path + '/train'
test_path = data_path + '/test'
labels_path = project_path + '/data/raw/archive/labeled_anomalies.csv'

print("SERIAL 18 loaded successfully.")
print("Project folder exists:", os.path.exists(project_path))
print("Train folder exists:", os.path.exists(train_path))
print("Test folder exists:", os.path.exists(test_path))
print("Labels file exists:", os.path.exists(labels_path))

Mounted at /content/drive
SERIAL 18 loaded successfully.
Project folder exists: True
Train folder exists: True
Test folder exists: True
Labels file exists: True


## Load A-8 Telemetry

We'll use the same A-8 setup so we can make a fair comparison with Isolation Forest.

In [2]:
channel = "A-8"

# Load training and test data
train_A8 = np.load(train_path + "/" + channel + ".npy")
test_A8 = np.load(test_path + "/" + channel + ".npy")

# Use the first column as the telemetry signal
train_telemetry_A8 = train_A8[:, 0]
test_telemetry_A8 = test_A8[:, 0]

print("Channel:", channel)
print("Training data shape:", train_A8.shape)
print("Test data shape:", test_A8.shape)
print("Training telemetry shape:", train_telemetry_A8.shape)
print("Test telemetry shape:", test_telemetry_A8.shape)

Channel: A-8
Training data shape: (762, 25)
Test data shape: (8375, 25)
Training telemetry shape: (762,)
Test telemetry shape: (8375,)


## Prepare Data for One-Class SVM

One-Class SVM also expects a 2D feature matrix.

We'll start with the same simple raw telemetry-only baseline so our comparison with Isolation Forest is fair.

In [3]:
# Reshape telemetry into 2D feature matrices
X_train_A8 = train_telemetry_A8.reshape(-1, 1)
X_test_A8 = test_telemetry_A8.reshape(-1, 1)

print("Training features shape:", X_train_A8.shape)
print("Test features shape:", X_test_A8.shape)

Training features shape: (762, 1)
Test features shape: (8375, 1)


## Train One-Class SVM

Now we'll train One-Class SVM using only the training data.

We'll start with:

kernel="rbf" → can capture nonlinear patterns.
gamma="scale" → lets scikit-learn choose a reasonable kernel width.
nu=0.05 → allows approximately 5% of the training observations to be treated as outliers.

Importantly, we are not using the test anomaly labels to train or tune the model.

In [4]:
# Create the One-Class SVM model
one_class_svm = OneClassSVM(
    kernel="rbf",
    gamma="scale",
    nu=0.05
)

# Train using normal training telemetry
one_class_svm.fit(X_train_A8)

print("One-Class SVM training completed.")

One-Class SVM training completed.


## Predict Anomalies

Now we'll run the trained One-Class SVM on the 8,375 A-8 test points.

One-Class SVM returns:

1 → normal

-1 → anomaly

We'll convert that to our project format:

0 → normal

1 → anomaly

In [5]:
# Predict anomalies in the test data
svm_predictions = one_class_svm.predict(X_test_A8)

# Convert:
# 1 = normal → 0
# -1 = anomaly → 1
y_pred_svm_A8 = (svm_predictions == -1).astype(int)

print("One-Class SVM prediction completed.")
print("Predicted normal:", (y_pred_svm_A8 == 0).sum())
print("Predicted anomalies:", (y_pred_svm_A8 == 1).sum())

One-Class SVM prediction completed.
Predicted normal: 1349
Predicted anomalies: 7026


## Create Ground-Truth Labels

In [6]:
# Load ground-truth anomaly labels
labels_df = pd.read_csv(labels_path)
A8_info = labels_df[labels_df["chan_id"] == channel].iloc[0]

# Create binary ground-truth labels
y_true_A8 = np.zeros(len(test_telemetry_A8), dtype=int)

anomaly_start = 4569
anomaly_end = 8374

y_true_A8[anomaly_start:anomaly_end + 1] = 1

print("Ground-truth labels created.")
print("Ground-truth normal:", (y_true_A8 == 0).sum())
print("Ground-truth anomalies:", (y_true_A8 == 1).sum())

Ground-truth labels created.
Ground-truth normal: 4569
Ground-truth anomalies: 3806


Now we have both:

Actual labels: 4,569 normal / 3,806 anomaly

One-Class SVM predictions: 1,349 normal / 7,026 anomaly

## Calculate Precision, Recall & F1

In [7]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Calculate evaluation metrics
precision_svm = precision_score(
    y_true_A8,
    y_pred_svm_A8,
    zero_division=0
)

recall_svm = recall_score(
    y_true_A8,
    y_pred_svm_A8,
    zero_division=0
)

f1_svm = f1_score(
    y_true_A8,
    y_pred_svm_A8,
    zero_division=0
)

print("One-Class SVM Evaluation")
print("-------------------------")
print("Precision:", precision_svm)
print("Recall:", recall_svm)
print("F1-score:", f1_svm)

One-Class SVM Evaluation
-------------------------
Precision: 0.3551095929405067
Recall: 0.6555438780872307
F1-score: 0.46067208271787297


| Metric    | One-Class SVM |
| --------- | ------------: |
| Precision |    **0.3551** |
| Recall    |    **0.6555** |
| F1-score  |    **0.4607** |


What this means
Recall 65.55% → it detected about 66% of the actual anomalies. That's much better than Global Z-score and Rolling Z-score on A-8.

Precision 35.51% → many of its anomaly predictions are false alarms.

F1 46.07% → overall, it is better than the statistical baselines we tested, but it's still not a practical detector yet.


#calculate the confusion matrix:

In [8]:
from sklearn.metrics import confusion_matrix

cm_svm = confusion_matrix(y_true_A8, y_pred_svm_A8)

tn_svm, fp_svm, fn_svm, tp_svm = cm_svm.ravel()

print("One-Class SVM Confusion Matrix")
print("------------------------------")
print("True Negatives:", tn_svm)
print("False Positives:", fp_svm)
print("False Negatives:", fn_svm)
print("True Positives:", tp_svm)

One-Class SVM Confusion Matrix
------------------------------
True Negatives: 38
False Positives: 4531
False Negatives: 1311
True Positives: 2495


Now we can see why the precision is low.

|                    | Predicted Normal | Predicted Anomaly |
| ------------------ | ---------------: | ----------------: |
| **Actual Normal**  |               38 |         **4,531** |
| **Actual Anomaly** |            1,311 |         **2,495** |

One-Class SVM detects anomalies reasonably well, but its high false-positive rate makes the raw 1D version unsuitable for practical detection on A-8.


###calculate the False Positive Rate and False Negative Rate.

In [9]:
false_positive_rate_svm = fp_svm / (fp_svm + tn_svm)
false_negative_rate_svm = fn_svm / (fn_svm + tp_svm)

print("One-Class SVM Error Rates")
print("-------------------------")
print("False Positive Rate:", false_positive_rate_svm)
print("False Negative Rate:", false_negative_rate_svm)

One-Class SVM Error Rates
-------------------------
False Positive Rate: 0.9916830816371197
False Negative Rate: 0.3444561219127693


The baseline One-Class SVM achieved moderate anomaly detection capability on A-8, but its extremely high false-positive rate (99.17%) makes the raw 1D implementation unsuitable for practical anomaly detection.

##save the complete One-Class SVM result

In [10]:
id="svm-save"
results_path = project_path + '/results'

svm_result = pd.DataFrame([{
    "channel": channel,
    "method": "One-Class SVM",
    "kernel": "rbf",
    "gamma": "scale",
    "nu": 0.05,
    "precision": precision_svm,
    "recall": recall_svm,
    "f1_score": f1_svm,
    "false_positive_rate": false_positive_rate_svm,
    "false_negative_rate": false_negative_rate_svm,
    "true_negatives": tn_svm,
    "false_positives": fp_svm,
    "false_negatives": fn_svm,
    "true_positives": tp_svm
}])

svm_result.to_csv(
    results_path + "/one_class_svm_evaluation_A8.csv",
    index=False
)

print("One-Class SVM evaluation saved successfully.")
print(results_path + "/one_class_svm_evaluation_A8.csv")

One-Class SVM evaluation saved successfully.
/content/drive/MyDrive/Spacecraft-Anomaly-Detection/results/one_class_svm_evaluation_A8.csv


| Method            | Precision |    Recall |        F1 |       FPR |
| ----------------- | --------: | --------: | --------: | --------: |
| Global Z-score    |     0.000 |     0.000 |     0.000 |     0.000 |
| Rolling Z-score   |     0.582 |     0.027 |     0.052 |     0.016 |
| Isolation Forest  |     0.454 |     1.000 |     0.625 |     1.000 |
| **One-Class SVM** | **0.355** | **0.656** | **0.461** | **0.992** |


The important point is that F1 alone is not enough. Isolation Forest has the highest F1 here, but its 100% false-positive rate makes it unusable in this raw baseline form. One-Class SVM has the same fundamental problem.

##github commit


In [11]:
%cd /content/drive/MyDrive/Spacecraft-Anomaly-Detection

!git status

/content/drive/MyDrive/Spacecraft-Anomaly-Detection
Refresh index: 100% (25/25), done.
On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   notebooks/13_evaluate_global_zscore.ipynb
	modified:   notebooks/14_rolling_zscore.ipynb
	modified:   notebooks/15_compare_statistical_methods.ipynb
	modified:   notebooks/16_isolation_forest.ipynb
	modified:   notebooks/17_evaluate_isolation_forest.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	notebooks/18_one_class_svm.ipynb
	results/one_class_svm_evaluation_A8.csv

no changes added to commit (use "git add" and/or "git commit -a")


In [12]:
!git add notebooks/18_one_class_svm.ipynb results/one_class_svm_evaluation_A8.csv

!git commit -m "Complete SERIAL 18 One-Class SVM"

Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@07ac6cb08d53.(none)')


In [13]:
!git config --global user.name "Amit Chandra Das"
!git config --global user.email "arickroy0@gmail.com"

!git commit -m "Complete SERIAL 18 One-Class SVM"

[main d4d6106] Complete SERIAL 18 One-Class SVM
 2 files changed, 3 insertions(+)
 create mode 100644 notebooks/18_one_class_svm.ipynb
 create mode 100644 results/one_class_svm_evaluation_A8.csv


In [14]:
!git push origin main

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 4.89 KiB | 1001.00 KiB/s, done.
Total 6 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/Amit-Chandra-Das/spacecraft-anomaly-detection.git
   1198d43..d4d6106  main -> main
